# **ELM Model Quantization and Inference**

## Standalone Notebook for Neonatal RDS Prediction

## This notebook loads a pre-trained ELM model, quantizes its weights from float32 to int8,
## and compares performance before and after quantization."

## 1. Import libraries

In [53]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_curve, auc

## 2. Load Original ELM Model

In [54]:
# Load the saved ELM model
print("Loading ELM model...")
elm_model = joblib.load("elm_model.pkl")

# Extract components - all these keys exist in your model
w = elm_model['w']
b = elm_model['b']
beta = elm_model['beta']
scaler = elm_model['scaler']
feature_names = elm_model['feature_names']
original_accuracy = elm_model['accuracy']

print(f" Input weights (w) shape: {w.shape}")
print(f" Bias (b) shape: {b.shape}")
print(f" Output weights (beta) shape: {beta.shape}")
print(f" Original accuracy: {original_accuracy:.4f}")

Loading ELM model...
 Input weights (w) shape: (10, 950)
 Bias (b) shape: (1, 950)
 Output weights (beta) shape: (950,)
 Original accuracy: 0.9832


## 3. Load Test Data

In [55]:
# Load and prepare test data
print("\nLoading test data...")
df = pd.read_csv('../../clinical_data/neonatal_processed.csv')

X = df.drop("primary_outcome", axis=1)
y = df["primary_outcome"]

# Scale using saved scaler
X_scaled = scaler.transform(X)

# Use same test split as original (random_state=42, test_size=0.25)
_, X_test, _, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

print(f"Test set size: {len(X_test)}")
print(f"Positive cases in test set: {y_test.sum()}")
print(f"Negative cases in test set: {len(y_test) - y_test.sum()}")


Loading test data...
Test set size: 7500
Positive cases in test set: 204
Negative cases in test set: 7296


## 4. Define Quantization Functions

In [56]:
def quantize_float32_to_int8(weights):
    
    # Quantize float32 weights to int8.
    # Returns: (quantized_weights, scale, zero_point)
    
    w_min = weights.min()
    w_max = weights.max()
    
    # Int8 range: -128 to 127
    scale = (w_max - w_min) / 255
    zero_point = np.round(-128 - w_min / scale).astype(np.int8)
    
    # Quantize
    weights_quantized = np.round(weights / scale + zero_point).astype(np.int8)
    
    return weights_quantized, scale, zero_point

def dequantize_int8_to_float32(weights_quantized, scale, zero_point):
    # Convert int8 back to float32 for inference
    return (weights_quantized.astype(np.float32) - zero_point.astype(np.float32)) * scale

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def predict_elm_original(X, w, b, beta, threshold=0.35):
    # Original ELM prediction (float32)
    h = sigmoid(np.dot(X, w) + b)
    y_prob = np.dot(h, beta).flatten()
    y_pred = (y_prob >= threshold).astype(int)
    return y_pred, y_prob

def predict_elm_quantized(X, w_q, w_scale, w_zp, b_q, b_scale, b_zp, beta_q, beta_scale, beta_zp, threshold=0.35):
    # ELM prediction with quantized weights (dequantizes on the fly)
    # Dequantize weights
    w = dequantize_int8_to_float32(w_q, w_scale, w_zp)
    b = dequantize_int8_to_float32(b_q, b_scale, b_zp)
    beta = dequantize_int8_to_float32(beta_q, beta_scale, beta_zp)
    
    # Forward pass
    h = sigmoid(np.dot(X, w) + b)
    y_prob = np.dot(h, beta).flatten()
    y_pred = (y_prob >= threshold).astype(int)
    
    return y_pred, y_prob


## 5. Quantize All Three Weight Matrices

In [57]:
print("=" * 60)
print("QUANTIZING ELM WEIGHTS")
print("=" * 60)

# Quantize input weights (w)
w_q, w_scale, w_zp = quantize_float32_to_int8(w)
print(f"\nInput weights (w):")
print(f"  Shape: {w.shape}")
print(f"  Original size: {w.nbytes/1024:.2f} KB (float32)")
print(f"  Quantized size: {w_q.nbytes/1024:.2f} KB (int8)")
print(f"  Compression: {w.nbytes/w_q.nbytes:.1f}x")

# Quantize bias (b)
b_q, b_scale, b_zp = quantize_float32_to_int8(b)
print(f"\nBias (b):")
print(f"  Shape: {b.shape}")
print(f"  Original size: {b.nbytes/1024:.2f} KB (float32)")
print(f"  Quantized size: {b_q.nbytes/1024:.2f} KB (int8)")
print(f"  Compression: {b.nbytes/b_q.nbytes:.1f}x")

# Quantize output weights (beta)
beta_q, beta_scale, beta_zp = quantize_float32_to_int8(beta)
print(f"\nOutput weights (beta):")
print(f"  Shape: {beta.shape}")
print(f"  Original size: {beta.nbytes/1024:.2f} KB (float32)")
print(f"  Quantized size: {beta_q.nbytes/1024:.2f} KB (int8)")
print(f"  Compression: {beta.nbytes/beta_q.nbytes:.1f}x")

# Total size calculation
original_total = w.nbytes + b.nbytes + beta.nbytes
quantized_total = w_q.nbytes + b_q.nbytes + beta_q.nbytes

print(f"\n{'='*60}")
print(f"TOTAL ORIGINAL SIZE: {original_total/1024:.2f} KB")
print(f"TOTAL QUANTIZED SIZE: {quantized_total/1024:.2f} KB")
print(f"OVERALL COMPRESSION: {original_total/quantized_total:.2f}x")
print(f"{'='*60}")

QUANTIZING ELM WEIGHTS

Input weights (w):
  Shape: (10, 950)
  Original size: 74.22 KB (float32)
  Quantized size: 9.28 KB (int8)
  Compression: 8.0x

Bias (b):
  Shape: (1, 950)
  Original size: 7.42 KB (float32)
  Quantized size: 0.93 KB (int8)
  Compression: 8.0x

Output weights (beta):
  Shape: (950,)
  Original size: 7.42 KB (float32)
  Quantized size: 0.93 KB (int8)
  Compression: 8.0x

TOTAL ORIGINAL SIZE: 89.06 KB
TOTAL QUANTIZED SIZE: 11.13 KB
OVERALL COMPRESSION: 8.00x


## 6. Compare Original vs Quantized Performance

In [58]:
print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)

# Original ELM predictions
y_pred_orig, y_prob_orig = predict_elm_original(X_test, w, b, beta)
acc_orig = accuracy_score(y_test, y_pred_orig)

# Quantized ELM predictions
y_pred_quant, y_prob_quant = predict_elm_quantized(
    X_test, 
    w_q, w_scale, w_zp, 
    b_q, b_scale, b_zp, 
    beta_q, beta_scale, beta_zp
)
acc_quant = accuracy_score(y_test, y_pred_quant)

print(f"\nOriginal ELM Accuracy:   {acc_orig:.4f} ({acc_orig*100:.2f}%)")
print(f"Quantized ELM Accuracy:  {acc_quant:.4f} ({acc_quant*100:.2f}%)")
print(f"Accuracy Drop:           {(acc_orig - acc_quant)*100:.4f}%")

# PR-AUC comparison
precision_orig, recall_orig, _ = precision_recall_curve(y_test, y_prob_orig)
pr_auc_orig = auc(recall_orig, precision_orig)

precision_quant, recall_quant, _ = precision_recall_curve(y_test, y_prob_quant)
pr_auc_quant = auc(recall_quant, precision_quant)

print(f"\nOriginal PR-AUC:   {pr_auc_orig:.4f}")
print(f"Quantized PR-AUC:  {pr_auc_quant:.4f}")
print(f"PR-AUC Drop:       {(pr_auc_orig - pr_auc_quant)*100:.4f}%")

PERFORMANCE COMPARISON

Original ELM Accuracy:   0.9832 (98.32%)
Quantized ELM Accuracy:  0.6447 (64.47%)
Accuracy Drop:           33.8533%

Original PR-AUC:   0.7610
Quantized PR-AUC:  0.5575
PR-AUC Drop:       20.3475%


## 7. Save Quantized Model

In [59]:
# Create quantized model dictionary
elm_quantized = {
    'w_q': w_q,
    'w_scale': w_scale,
    'w_zero_point': w_zp,
    'b_q': b_q,
    'b_scale': b_scale,
    'b_zero_point': b_zp,
    'beta_q': beta_q,
    'beta_scale': beta_scale,
    'beta_zero_point': beta_zp,
    'scaler': scaler,
    'input_size': elm_model['input_size'],
    'hidden_neurons': elm_model['hidden_neurons'],
    'output_size': elm_model['output_size'],
    'feature_names': feature_names,
    'original_accuracy': acc_orig,
    'quantized_accuracy': acc_quant,
    'accuracy_drop_percent': (acc_orig - acc_quant) * 100,
    'original_size_kb': original_total / 1024,
    'quantized_size_kb': quantized_total / 1024,
    'compression_ratio': original_total / quantized_total
}

# Save to file
joblib.dump(elm_quantized, "elm_model_quantized.pkl")
print("Quantized ELM model saved to 'elm_model_quantized.pkl'")

# Show file size
file_size = os.path.getsize("elm_model_quantized.pkl") / 1024
print(f"Saved file size: {file_size:.2f} KB")

Quantized ELM model saved to 'elm_model_quantized.pkl'
Saved file size: 13.24 KB


## 8. Summary Table for Report

In [60]:
print("\n" + "=" * 70)
print("SUMMARY FOR TECHNICAL REPORT")
print("=" * 70)

print(f"""
| Metric                    | Original (float32) | Quantized (int8) |
|---------------------------|--------------------|------------------|
| Model Size                | {original_total/1024:.2f} KB           | {quantized_total/1024:.2f} KB         |
| Compression Ratio         | -                  | {original_total/quantized_total:.2f}x           |
| Accuracy                  | {acc_orig:.4f} ({acc_orig*100:.2f}%)    | {acc_quant:.4f} ({acc_quant*100:.2f}%)   |
| PR-AUC                    | {pr_auc_orig:.4f}             | {pr_auc_quant:.4f}             |
| Accuracy Drop             | -                  | {(acc_orig - acc_quant)*100:.4f}%          |
""")

print("=" * 70)


SUMMARY FOR TECHNICAL REPORT

| Metric                    | Original (float32) | Quantized (int8) |
|---------------------------|--------------------|------------------|
| Model Size                | 89.06 KB           | 11.13 KB         |
| Compression Ratio         | -                  | 8.00x           |
| Accuracy                  | 0.9832 (98.32%)    | 0.6447 (64.47%)   |
| PR-AUC                    | 0.7610             | 0.5575             |
| Accuracy Drop             | -                  | 33.8533%          |

